# Preprocessing: shared news inputs for static and contextual representations

Prepare **TF-IDF, pretrained Word2Vec, and transformer inputs** for binary next-fixing USD/IDR classification. This notebook performs deterministic preparation only; vocabulary, IDF, SVD, scalers, embeddings, and prediction models are fitted or loaded later within the experiment protocol.

Text comes exclusively from the processed news CSV, never from raw scraping or the old notebooks. The requested `data/preprocessed/news_clean.csv` is used if present; the current repository only contains `data/processed/news_clean.csv`, which is explicitly reported as the fallback. Both the resolved path and its hash are recorded.

Read [the ablation protocol](../rules/ablation_protocol.md), [binary protocol](../rules/binary_classification_protocol.md), and [README](README.md). The binary protocol clarifies timing and gap assumptions in the original preprocessing rules. No particular model or longer-gap variance advantage is assumed.

**Current limitation:** news has dates, not verified publication timestamps. The default is a delayed date-only timing sensitivity dataset, not a verified strict-forecast dataset. Keyword geopolitical triage is a review aid; by default it does not silently redefine the collected corpus.

In [1]:
from pathlib import Path
import importlib
import json
import platform
import sys

import numpy as np
import pandas as pd

# Locate the repository from either its root or this notebook directory.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/processed/usd_idr_jisdor_clean.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the project root or a directory beneath it.")
NOTEBOOK_DIR = ROOT / "deira-work/preprocessing"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))
import preprocessing_utils as prep
importlib.reload(prep)
print(f"Repository: {ROOT}")
print(f"Python {platform.python_version()} | pandas {pd.__version__} | NumPy {np.__version__}")

Repository: /home/lenovo/UGM/assignment/semester_5/nlp/nlp-geopolitical-news-usd-prediction
Python 3.12.14 | pandas 2.2.3 | NumPy 2.3.5


## 1. Configuration and provenance

Freeze these choices before comparing representations. `MAX_BODY_WORDS` defines a common raw span, not a model token limit. Word2Vec casing should match its pretrained vocabulary. Review files are optional user inputs and are never overwritten.

`audit_only` retains quality-passing records regardless of the keyword geopolitical status; explicit human exclusions still apply. Use `human_approved` for a manually validated geopolitical corpus. `rule_screened` is only a provisional keyword subset. Do not pick a corpus policy using test scores.

In [2]:
REQUESTED_NEWS_PATH = ROOT / "data/preprocessed/news_clean.csv"
EXISTING_NEWS_PATH = ROOT / "data/processed/news_clean.csv"
NEWS_PATH = REQUESTED_NEWS_PATH if REQUESTED_NEWS_PATH.is_file() else EXISTING_NEWS_PATH
FX_PATH = ROOT / "data/processed/usd_idr_jisdor_clean.csv"
if not NEWS_PATH.is_file():
    raise FileNotFoundError(f"Neither requested nor existing processed-news input exists: {REQUESTED_NEWS_PATH}")

ALIGNMENT_MODE = "date_delay_sensitivity"  # or strict_forecast with verified evidence
GEOPOLITICAL_POLICY = "audit_only"        # audit_only / rule_screened / human_approved
MAX_BODY_WORDS = 180
WORD2VEC_LOWERCASE = False
INCLUDE_RULE_REVIEW = False              # only relevant to rule_screened
TEST_FRACTION = 0.20
N_FOLDS = 3
REVIEW_PATH = NOTEBOOK_DIR / "reviews/article_review.csv"
SCHEDULE_PATH = NOTEBOOK_DIR / "reviews/jisdor_availability.csv"
OUTPUT_DIR = NOTEBOOK_DIR / "outputs" / f"{ALIGNMENT_MODE}__{GEOPOLITICAL_POLICY}"

print(f"Requested text path exists: {REQUESTED_NEWS_PATH.is_file()}")
print(f"Resolved text source: {NEWS_PATH.relative_to(ROOT)}")
if NEWS_PATH != REQUESTED_NEWS_PATH:
    print("Explicit fallback: requested data/preprocessed path is absent in this snapshot.")
print(f"Mode: {ALIGNMENT_MODE}; corpus policy: {GEOPOLITICAL_POLICY}")

Requested text path exists: False
Resolved text source: data/processed/news_clean.csv
Explicit fallback: requested data/preprocessed path is absent in this snapshot.
Mode: date_delay_sensitivity; corpus policy: audit_only


In [3]:
raw_news = pd.read_csv(NEWS_PATH, keep_default_na=False)
raw_fx = pd.read_csv(FX_PATH)
source_paths = [NEWS_PATH, FX_PATH]
input_hashes = {str(p.relative_to(ROOT)): prep.sha256(p) for p in source_paths}
print(f"News: {len(raw_news)} records; JISDOR: {len(raw_fx)} fixing rows")
print("News columns:", ", ".join(raw_news.columns))
print(json.dumps(input_hashes, indent=2))
print("Legacy Aligned_Date is retained for audit only, never used as an availability cutoff.")

News: 435 records; JISDOR: 1202 fixing rows
News columns: Date, Aligned_Date, Title, Source, URL, Event, Keyword, Category, Text_clean
{
  "data/processed/news_clean.csv": "81ff1e273e49d678fc86f0377c78b08be91004a3a248e660d58d6ec73176c4b1",
  "data/processed/usd_idr_jisdor_clean.csv": "faf181306cdb65c1d74f913ef698fe45008e721969372bb17f79175d20fe87a0"
}
Legacy Aligned_Date is retained for audit only, never used as an availability cutoff.


## 2. Shared cleaning, quality controls, and review

Normalize Unicode, decode HTML entities, remove explicit footer/credit paragraphs, and deduplicate repeated long paragraphs. Keep the headline and first 180 cleaned body words for every representation. This reduces exposure to related-story cards but cannot recover the original historical body.

Confirmed later-content annotations are reused only with an exact source-hash match. Those records remain quarantined even if their visible lead looks plausible. Future-date mentions, URL/date disagreement, and encoding problems enter review; they are not proof of historical leakage by themselves. Short and exact-duplicate bodies are excluded, keeping the earliest record. No target outcome is used for filtering.

Manual review fields: `geo_decision` (include/exclude/review), `review_note`, `published_at`, `available_at`, `source_timezone`, `timestamp_verified`, `body_asof_verified`, and `evidence`. Times must carry an offset or an explicit timezone. A manual include may resolve heuristic flags, but cannot override confirmed later content or exact duplicates.

In [4]:
known_future_rows, audit_message = prep.audit_future_ids(ROOT, NEWS_PATH)
prepared = prep.prepare_news(raw_news, max_body_words=MAX_BODY_WORDS,
                            word2vec_lowercase=WORD2VEC_LOWERCASE,
                            known_future_ids=known_future_rows)
reviewed = prep.apply_reviews(prepared, REVIEW_PATH)
selected = prep.choose_candidates(reviewed, geopolitical_policy=GEOPOLITICAL_POLICY,
                                 include_rule_review=INCLUDE_RULE_REVIEW)
assert selected.article_id.is_unique
assert len(selected) == len(raw_news)
assert not selected.loc[selected.known_temporal_contamination, "selected_for_alignment"].any()
print(audit_message)
print("Human review loaded:", REVIEW_PATH.is_file())
print("Keyword triage (not gold labels):")
print(selected.geo_status.value_counts().to_string())
print("Selection reasons:")
print(selected.selection_reason.value_counts().to_string())
print("Retained for alignment:", int(selected.selected_for_alignment.sum()))
print("Articles with removed paragraphs:", int(selected.removed_paragraphs.gt(0).sum()))
print("Residual boilerplate flags (inspect, not automatically removed):", int(selected.boilerplate_remaining.sum()))

Applied 10 confirmed temporal flags after an exact input-hash match.
Human review loaded: False
Keyword triage (not gold labels):
geo_status
exclude      288
review        98
candidate     49
Selection reasons:
selection_reason
quality_pass_or_manually_approved    414
known_later_content_quarantine        10
quality_review_required                8
insufficient_main_text                 2
exact_content_duplicate                1
Retained for alignment: 414
Articles with removed paragraphs: 52
Residual boilerplate flags (inspect, not automatically removed): 25


## 3. Representation-specific views

The shared semantic span is identical. Static tokenization differs from the pretrained transformer's tokenizer:

| View | Preparation here | Fit/load later |
|---|---|---|
| TF-IDF | Lowercase tokens; retain negation, names, numbers; no stemming/stopword deletion | Training-only vocabulary/IDF and SVD; transform other splits |
| Word2Vec | Case-preserving tokens by default; preserve meaning-bearing words | Pretrained vectors; report OOV and casing coverage |
| Transformer | Natural text with punctuation and casing | Checkpoint tokenizer; masked token mean excluding special/padding tokens |

Static tokenization removes most punctuation, so it is not an exact textual copy. Contractions preserve negation (`won't` → `will not`). No stemming or lemmatization is required for the baseline. Model-specific token truncation must be documented downstream. Mean word vectors form an article vector for Word2Vec; mean article vectors form a window vector for all Level 2 models. Keep individual article vectors for Level 3 attention.

In [5]:
example = "The U.S. will not cut rates; Bank Indonesia won't ignore a 5.25% shock."
print("Natural text:", prep.clean_plain_text(example))
print("TF-IDF tokens:", prep.static_tokens(example, lowercase=True))
print("Word2Vec tokens:", prep.static_tokens(example, lowercase=WORD2VEC_LOWERCASE))
assert "not" in prep.static_tokens(example, lowercase=True)
assert "5.25%" in prep.static_tokens(example, lowercase=True)
assert (selected.text_transformer == selected.text_shared).all()
assert (selected.text_tfidf == selected.tokens_tfidf.map(" ".join)).all()
view_preview = selected.loc[selected.selected_for_alignment,
                            ["source_row", "title_main", "main_word_count"]].head(5)
print(view_preview.to_string(index=False))

Natural text: The US will not cut rates; Bank Indonesia won't ignore a 5.25% shock.
TF-IDF tokens: ['the', 'us', 'will', 'not', 'cut', 'rates', 'bank', 'indonesia', 'will', 'not', 'ignore', 'a', '5.25%', 'shock']
Word2Vec tokens: ['The', 'US', 'will', 'not', 'cut', 'rates', 'Bank', 'Indonesia', 'will', 'not', 'ignore', 'a', '5.25%', 'shock']
 source_row                                                                                                 title_main  main_word_count
          0                                  Victorious Taliban focus on governing a wary populace after US withdrawal             1135
          2 El Salvador could be hit by a crypto crash after it adopts bitcoin – and would be powerless to do anything              782
          3                                 Lebanon gets a new government, its first since last year deadly port blast              533
          4                                                     Lebanon richest man becomes country new Prime Mi

## 4. Original JISDOR calendar and binary labels

One row per observed fixing, with no weekend interpolation. Features use only the current and earlier fixings. `log_return_t` is the most recently observed return; `return_lag1` is one additional fixing older. Moving-average ratios and volatility windows are in fixing observations. RSI uses a documented simple 14-observation gain/loss mean (not Wilder smoothing).

Calculate `log_return_next = log(J_next / J_t)` on the **full calendar**, then set UP=1, DOWN=0, and FLAT/unavailable=`NA`. UP is rupiah depreciation. Preserve FLAT and warmup rows in the calendar, even though they are not supervised samples. `realized_target_gap_days` and target weekday are evaluation diagnostics, never default predictors. Longer-gap variance is to be checked empirically.

In [6]:
calendar_frame = prep.build_calendar(raw_fx)
assert len(calendar_frame) == len(raw_fx)
assert not calendar_frame.Date.duplicated().any()
assert calendar_frame.Date.dt.dayofweek.lt(5).all()
assert calendar_frame.loc[calendar_frame.target_status.isin(["FLAT", "UNAVAILABLE"]), "y"].isna().all()
print("Fixing range:", calendar_frame.Date.min().date(), "to", calendar_frame.Date.max().date())
print("Adjacent target statuses before warmup masking:")
print(calendar_frame.target_status.value_counts().to_string())
print("Eligible binary rows:", int(calendar_frame.eligible_binary.sum()))
print("Historical input-gap distribution:")
print(calendar_frame.input_gap_days.value_counts().sort_index().to_string())
print("Feature whitelist:", ", ".join(prep.MARKET_FEATURES))

Fixing range: 2021-09-01 to 2026-09-01
Adjacent target statuses before warmup masking:
target_status
UP             665
DOWN           523
FLAT            13
UNAVAILABLE      1
Eligible binary rows: 1168
Historical input-gap distribution:
input_gap_days
1.0     922
2.0      18
3.0     223
4.0      18
5.0      13
6.0       2
8.0       2
11.0      2
12.0      1
Feature whitelist: log_return_t, return_lag1, return_lag2, return_lag3, return_lag5, volatility_5, volatility_20, ma_ratio_5, ma_ratio_20, momentum_5, momentum_10, rsi_14, input_gap_days, decision_weekday


## 5. Align by information availability

**Default date-delay sensitivity:** map a date-only article to the first fixing with a strictly later date. Friday/Saturday/Sunday news enters Monday if it is the next fixing; Monday-dated news enters Tuesday. This avoids asserting same-day availability. It still assumes the source calendar date is meaningful and cannot certify historical timestamps or stored body versions.

**Optional strict forecast:** require a verified publication/decision schedule for every fixing, plus verified article availability and body-version evidence. Use windows `(previous decision, current decision]`; a timestamp exactly at a decision belongs to that decision. Decision is at/after actual JISDOR publication. Convert known article times to WIB. Do not assume one historical fixing/publication hour for every year.

In either mode, the aligned date is the **input/decision anchor**; its label is the following fixing. This notebook does not build an explanatory same-window dataset. Early articles before the first cutoff enter the first observed window; no observed news precedes this snapshot's first date, and warmup rows remain masked.

In [7]:
schedule = None
if ALIGNMENT_MODE == "strict_forecast":
    if not SCHEDULE_PATH.is_file():
        raise FileNotFoundError("Strict mode needs reviews/jisdor_availability.csv; generate and verify the template in default mode first.")
    schedule = prep.read_verified_schedule(SCHEDULE_PATH, calendar_frame)

aligned = prep.align_articles(selected, calendar_frame, mode=ALIGNMENT_MODE, schedule=schedule)
if ALIGNMENT_MODE == "strict_forecast" and not aligned.anchor_date.notna().any():
    raise ValueError("No trusted, aligned news remains. Verify article availability and historical body evidence before exporting strict text inputs.")
if ALIGNMENT_MODE == "date_delay_sensitivity":
    used = aligned.loc[aligned.anchor_date.notna()]
    assert (used.anchor_date > used.source_date).all()
print(aligned.alignment_status.value_counts().to_string())
print("Aligned articles:", int(aligned.anchor_date.notna().sum()))

alignment_status
date_only_delayed_assumption    412
not_selected                     21
after_last_available_fixing       2
Aligned articles: 412


## 6. Windows, empty-news masks, and fixed chronological splits

Keep all fixing rows, including days with zero news. Empty windows have empty lists, `n_berita = 0`, and `has_news = 0`, not invented neutral sentiment. Padding masks belong to the downstream batch builder. For all-empty samples, bypass attention before softmax; do not attend to an entirely masked key sequence.

Reserve the last 20% of fixing indices as the test period. Purge the adjacent development boundary and use three expanding chronological validation folds in development. A training label must end strictly before the first validation decision date. Fold eligibility uses only feature warmup and the predeclared FLAT policy; a date is not dropped because it lacks news. No learned transformation is fitted in this notebook.

In [8]:
daily, windows = prep.build_daily_tables(calendar_frame, aligned)
if schedule is not None:
    daily["decision_at_wib"] = schedule.decision_at_wib.to_numpy()
    daily["observed_window_hours"] = schedule.decision_at_wib.diff().dt.total_seconds() / 3600
else:
    daily["decision_at_wib"] = pd.NaT
    daily["observed_window_hours"] = np.nan  # Date gaps are not verified timestamp durations.
partitions, folds, fold_summary, final_fit = prep.temporal_manifests(
    daily, test_fraction=TEST_FRACTION, n_folds=N_FOLDS)
assert len(daily) == len(windows) == len(raw_fx)
assert windows.article_ids.map(len).eq(windows.n_berita).all()
assert windows.loc[windows.n_berita.eq(0), "texts_transformer"].map(len).eq(0).all()
print("All-calendar news-window counts:")
print(pd.Series({"empty": int(daily.n_berita.eq(0).sum()),
                 "singleton": int(daily.n_berita.eq(1).sum()),
                 "multi_article": int(daily.n_berita.gt(1).sum())}).to_string())
print("Development fold diagnostics:")
print(fold_summary.to_string(index=False))
test_dates = partitions.loc[partitions.partition.eq("test"), "Date"]
print("Reserved test calendar:", test_dates.min().date(), "to", test_dates.max().date())
print("Boundary purge:", partitions.loc[partitions.partition.eq("boundary_purge"), "Date"].dt.strftime("%Y-%m-%d").tolist())

All-calendar news-window counts:
empty            1000
singleton         120
multi_article      82
Development fold diagnostics:
 fold       role  rows first_date  last_date  up  down  days_with_news
    1      train   453 2021-09-29 2023-08-15 260   193              65
    1 validation   158 2023-08-21 2024-04-19  94    64              21
    2      train   612 2021-09-29 2024-04-19 355   257              86
    2 validation   159 2024-04-23 2024-12-12  81    78              32
    3      train   772 2021-09-29 2024-12-12 437   335             118
    3 validation   158 2024-12-16 2025-08-26  79    79              22
Reserved test calendar: 2025-08-29 to 2026-09-01
Boundary purge: ['2025-08-28']


## 7. Downstream fit boundary: select articles by training anchor

This example selects only first-fold training articles. Fit TF-IDF/SVD on these articles, not on the complete prepared table. For subsequent folds, regenerate the selection and refit from scratch. A frozen pretrained Word2Vec/transformer does not learn vocabulary from validation/test; any adapters or fine-tuning still use training only. Record checkpoint revisions, pretraining limits, OOV rates, and special-token pooling rules in the experiment.

Use the same window membership, outcome dates, features, and representation span in all ablations. The market feature scaler uses only the explicit whitelist and training rows. Do not include source labels, collection Event/Keyword, cleaning flags, next price, target return, or realized future gap as predictors.

In [9]:
first_train_dates = folds.loc[(folds.fold == 1) & folds.role.eq("train"), "Date"]
first_val_dates = folds.loc[(folds.fold == 1) & folds.role.eq("validation"), "Date"]
first_train_articles = aligned.loc[aligned.anchor_date.isin(first_train_dates)].copy()
assert not first_train_articles.anchor_date.isin(first_val_dates).any()
first_train_market = daily.loc[daily.Date.isin(first_train_dates), prep.MARKET_FEATURES]
assert first_train_market.notna().all().all()
print(f"First-fold training articles for vocabulary fitting: {len(first_train_articles)}")
print(f"First-fold scaler matrix: {first_train_market.shape}")
print("No vocabulary, IDF, SVD, scaler, or model has been fitted.")

First-fold training articles for vocabulary fitting: 190
First-fold scaler matrix: (453, 14)
No vocabulary, IDF, SVD, scaler, or model has been fitted.


## 8. Export reproducible inputs, review templates, and manifests

Generated artifacts are separated by timing mode and corpus policy. All article records remain auditable; only aligned selected articles enter windows. Templates are generated under outputs and never replace files under `reviews/`. To review, copy the template to the configured review path, add decisions/evidence, then rerun. Quality notes should explicitly resolve any flagged forward date or timestamp inconsistency before inclusion.

The code and input hashes let later experiments identify the exact preprocessing snapshot. Original source files and earlier notebooks are not modified. Test outcomes are not used to select filters or representations.

In [10]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
article_columns = ["article_id", "source_row", "row_fingerprint", "source_date", "Title", "Source", "URL", "Event",
                   "title_main", "body_main", "lead_main", "text_shared", "text_transformer", "text_tfidf",
                   "tokens_tfidf", "tokens_word2vec", "original_word_count", "main_word_count",
                   "removed_paragraphs", "removed_chars", "footer_cut_reason", "boilerplate_remaining",
                   "encoding_flag", "mentioned_later_dates", "future_date_heuristic", "known_temporal_contamination",
                   "url_date", "url_date_gap_days", "date_mismatch_review", "too_short", "duplicate_of",
                   "geo_status", "geo_reason", "quality_blocked", "quality_review", "effective_geo_status",
                   "selected_for_alignment", "selection_reason", "published_at_wib", "available_at_wib",
                   "timing_mode", "anchor_date", "alignment_status"]
article_columns += [c for c in prep.REVIEW_COLUMNS[1:] if c not in article_columns]
if "legacy_aligned_date" in aligned:
    article_columns.append("legacy_aligned_date")
prep.write_jsonl(aligned[article_columns], OUTPUT_DIR / "article_views.jsonl")
prep.write_jsonl(aligned.loc[aligned.anchor_date.notna(), article_columns], OUTPUT_DIR / "aligned_articles.jsonl")
prep.write_jsonl(windows, OUTPUT_DIR / "news_windows.jsonl")

review_columns = ["article_id", "source_row", "source_date", "Title", "URL", "geo_status", "geo_reason",
                  "quality_blocked", "quality_review", "known_temporal_contamination", "future_date_heuristic",
                  "date_mismatch_review", "encoding_flag", "boilerplate_remaining", "duplicate_of",
                  "selection_reason", "alignment_status", "lead_main"]
review_mask = aligned.quality_blocked | aligned.quality_review | aligned.boilerplate_remaining | ~aligned.geo_status.eq("candidate")
aligned.loc[review_mask, review_columns].to_csv(OUTPUT_DIR / "review_queue.csv", index=False)
aligned.loc[aligned.anchor_date.isna(), review_columns].to_csv(OUTPUT_DIR / "excluded_articles.csv", index=False)
template = aligned[["article_id", "source_date", "Title", *prep.REVIEW_COLUMNS[1:]]].copy()
template.to_csv(OUTPUT_DIR / "article_review_template.csv", index=False)
schedule_template = pd.DataFrame({"Date": calendar_frame.Date.dt.strftime("%Y-%m-%d"),
                                 "published_at_wib": "", "decision_at_wib": "", "verified": False, "source": ""})
schedule_template.to_csv(OUTPUT_DIR / "jisdor_availability_template.csv", index=False)
for filename, frame in [("daily_market.csv", daily), ("calendar_partitions.csv", partitions),
                        ("fold_manifest.csv", folds), ("fold_summary.csv", fold_summary),
                        ("final_fit_manifest.csv", final_fit)]:
    frame.to_csv(OUTPUT_DIR / filename, index=False)

contract = {
    "market_features": prep.MARKET_FEATURES,
    "news_availability_features": ["n_berita", "has_news"],
    "label": "y", "positive_class": "USD_IDR_UP__IDR_depreciation",
    "target_and_diagnostic_columns_never_predictors": prep.TARGET_COLUMNS + ["observed_window_hours"],
    "text_inputs": {"tfidf": "text_tfidf", "word2vec": "tokens_word2vec", "transformer": "text_transformer"},
    "article_metadata_is_audit_only": True,
    "fit_rule": "Select each fold's training dates, join articles by anchor_date, fit only there; transform other splits.",
    "empty_window": "Empty lists now; zero vectors, availability mask, and safe attention bypass downstream.",
    "window_aggregation": "Per-article vectors, then uniform article mean for Levels 1/2; retain article sequence for Level 3.",
}
config = {"alignment_mode": ALIGNMENT_MODE, "geopolitical_policy": GEOPOLITICAL_POLICY,
          "max_body_words": MAX_BODY_WORDS, "word2vec_lowercase": WORD2VEC_LOWERCASE,
          "include_rule_review": INCLUDE_RULE_REVIEW, "test_fraction": TEST_FRACTION, "n_folds": N_FOLDS}
extra_inputs = [p for p in [REVIEW_PATH, SCHEDULE_PATH if schedule is not None else None] if p is not None and p.is_file()]
if known_future_rows:
    extra_inputs += [ROOT / "deira-work/sanity-check/audit_metrics.json",
                     ROOT / "deira-work/sanity-check/future_date_review.csv"]
provenance_paths = [NOTEBOOK_DIR / "preprocessing_utils.py",
                    ROOT / "deira-work/rules/ablation_protocol.md",
                    ROOT / "deira-work/rules/binary_classification_protocol.md",
                    ROOT / "deira-work/rules/preprocessing_rules.md"]
manifest = {
    "requested_news_path": str(REQUESTED_NEWS_PATH.relative_to(ROOT)),
    "resolved_news_path": str(NEWS_PATH.relative_to(ROOT)),
    "input_sha256": input_hashes,
    "review_and_audit_sha256": {str(p.relative_to(ROOT)): prep.sha256(p) for p in extra_inputs},
    "code_and_protocol_sha256": {str(p.relative_to(ROOT)): prep.sha256(p) for p in provenance_paths},
    "config": config,
    "versions": {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__},
    "counts": {"input_articles": len(raw_news), "calendar_rows": len(daily),
               "selected_before_alignment": int(aligned.selected_for_alignment.sum()),
               "aligned_articles": int(aligned.anchor_date.notna().sum()),
               "days_with_news": int(daily.has_news.sum()),
               "eligible_binary_rows": int(daily.eligible_binary.sum()),
               "confirmed_temporal_quarantine": len(known_future_rows)},
    "strict_availability_verified": ALIGNMENT_MODE == "strict_forecast",
    "geopolitical_corpus_human_validated": GEOPOLITICAL_POLICY == "human_approved",
    "learned_transforms_fitted": False,
    "limitations": ["Date-only delay does not establish historical availability or body versions.",
                    "audit_only is not a manually validated geopolitical corpus.",
                    "Historical audit quarantines are retrospective curation.",
                    "Near-duplicate story/version clusters still require review before model evaluation."],
}
for filename, content in [("feature_contract.json", contract), ("pipeline_manifest.json", manifest)]:
    (OUTPUT_DIR / filename).write_text(json.dumps(content, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
for path in source_paths:
    assert prep.sha256(path) == input_hashes[str(path.relative_to(ROOT))], "Input unexpectedly changed."
print("Saved:", OUTPUT_DIR.relative_to(ROOT))
print(json.dumps(manifest["counts"], indent=2))
print("All source hashes unchanged. No learned preprocessing fitted.")

Saved: deira-work/preprocessing/outputs/date_delay_sensitivity__audit_only
{
  "input_articles": 435,
  "calendar_rows": 1202,
  "selected_before_alignment": 414,
  "aligned_articles": 412,
  "days_with_news": 202,
  "eligible_binary_rows": 1168,
  "confirmed_temporal_quarantine": 10
}
All source hashes unchanged. No learned preprocessing fitted.


## What remains for the experiment notebook

1. Resolve flagged article quality and validate the geopolitical corpus before final model comparisons. Inspect near-duplicate story versions in addition to exact body duplicates. Freeze shared IDs across representations.
2. Recover verified news/body availability and the historical JISDOR schedule before making a strict forecasting claim. Keep the current date-delay result explicitly labeled as a sensitivity experiment.
3. Fit TF-IDF/SVD and market scaling inside each training fold; load frozen pretrained Word2Vec/FinBERT/RoBERTa with recorded versions. Measure OOV rates and model-token truncation.
4. Implement the three ablation levels using common masks, dates, normalized pooling, and the feature contract. Preserve empty-news dates; distinguish article OOV from no collected news.

This notebook produces inputs, not embeddings, sentiment labels, trained models, or evidence that news predicts the exchange rate.